In [0]:
%python
%pip install yfinance

In [0]:
%python
# dml/02_carga_stg_historico_cotacoes_fiis.ipynb
# MAGIC %pip install yfinance pandas numpy # Garante a instalação do Yahoo Finance no compute Serverless

# %%
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# %%
# 1. Recupera os tickers ativos na tabela dim_fundo_imobiliario
catalogo = "product_dev"
schema = "financas"
tabela_dim = "dim_fundo_imobiliario"

print(f"Buscando tickers da tabela {catalogo}.{schema}.{tabela_dim}...")

# Lê a tabela do banco de dados usando Spark e joga para o Pandas
df_tickers = spark.sql(f"SELECT ticker FROM {catalogo}.{schema}.{tabela_dim}").toPandas()

# Extrai a lista de tickers e adiciona a extensão '.SA' necessária para o Yahoo Finance
tickers_b3 = df_tickers["ticker"].unique().tolist()
tickers_yf = [f"{t}.SA" for f in tickers_b3 for t in [f.strip()]]

print(f"Total de FIIs encontrados para monitoramento: {len(tickers_yf)}")

# %%
# 2. Faz o download em lote das séries históricas de 5 anos pelo yfinance
# Usar o lote com espaço (' ') evita que a API bloqueie seu IP e acelera em 10x o processo
if len(tickers_yf) == 0:
    print("Aviso: Nenhum ticker encontrado na dimensão de FIIs.")
else:
    print("Baixando histórico de 5 anos via Yahoo Finance API (isto pode levar de 30s a 1 min)...")
    
    # Baixa preços (OHLCV) e dividendos em lote
    dados_brutos = yf.download(
        tickers=" ".join(tickers_yf), 
        period="5y", 
        interval="1d", 
        group_by="ticker", 
        actions=True, # Traz a coluna de dividendos/proventos
        progress=True
    )

    print("Download concluído com sucesso!")

# %%
# 3. Transforma a matriz tridimensional do yfinance em um DataFrame bidimensional limpo (com tratamento defensivo)
lista_registros = []

for ticker_sa in tickers_yf:
    ticker_original = ticker_sa.replace(".SA", "")
    
    # Verifica se o ticker possui dados retornados pela API
    if ticker_sa in dados_brutos.columns.levels[0]:
        df_ticker = dados_brutos[ticker_sa]
        
        # Garante que as colunas essenciais existem e remove linhas em que o preço de fechamento é nulo ou NaN
        if "Close" in df_ticker.columns:
            df_ticker_limpo = df_ticker.dropna(subset=["Close"])
            
            for data_pregao, row in df_ticker_limpo.iterrows():
                # Tratamento defensivo de fallbacks: se o campo não existir ou for NaN, preenche de forma segura
                adj_close_val = float(row["Adj Close"]) if "Adj Close" in row and not np.isnan(row["Adj Close"]) else float(row["Close"])
                open_val = float(row["Open"]) if "Open" in row and not np.isnan(row["Open"]) else None
                high_val = float(row["High"]) if "High" in row and not np.isnan(row["High"]) else None
                low_val = float(row["Low"]) if "Low" in row and not np.isnan(row["Low"]) else None
                volume_val = int(row["Volume"]) if "Volume" in row and not np.isnan(row["Volume"]) else 0
                div_val = float(row["Dividends"]) if "Dividends" in row and not np.isnan(row["Dividends"]) else 0.0
                
                lista_registros.append({
                    "ticker": ticker_original,
                    "data_pregao": data_pregao.date(),
                    "preco_abertura": open_val,
                    "preco_maximo": high_val,
                    "preco_minimo": low_val,
                    "preco_fechamento": float(row["Close"]),
                    "preco_fechamento_ajustado": adj_close_val,
                    "volume_negociado": volume_val,
                    "proventos_pagos": div_val,
                    "data_carga": datetime.now()
                })

# Cria o DataFrame consolidado no Pandas
df_historico_pandas = pd.DataFrame(lista_registros)

# %%
# 4. Converte os dados limpos para Spark e cria a visualização temporária
print(f"Processando {len(df_historico_pandas)} linhas de histórico para inserção...")

# Converte o Pandas DataFrame para Spark DataFrame de forma nativa
spark_df_historico = spark.createDataFrame(df_historico_pandas)

# Cria a view temporária no Spark
spark_df_historico.createOrReplaceTempView("temp_historico_cotacoes")

# %%
# 5. Execução do INSERT OVERWRITE dinâmico na tabela de Staging de Histórico
tabela_destino = "stg_historico_cotacoes_fiis"

qry_insert_dim_fiis = f"""
  INSERT OVERWRITE {catalogo}.{schema}.{tabela_destino}
  SELECT 
    ticker,
    data_pregao,
    preco_abertura,
    preco_maximo,
    preco_minimo,
    preco_fechamento,
    preco_fechamento_ajustado,
    volume_negociado,
    proventos_pagos,
    data_carga
  FROM temp_historico_cotacoes
"""

print(f"Gravando dados de cotações em: {catalogo}.{schema}.{tabela_destino}...")

# Executa o overwrite atômico de séries temporais
spark.sql(qry_insert_dim_fiis)

print("✅ Carga das séries históricas de 5 anos concluída com SUCESSO!")